In [ ]:
import os
os.environ["MPLBACKEND"] = "agg"

import cv2
import math
import numpy as np
import mediapipe as mp

from IPython.display import display, HTML
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


live_handle = None
summary_handle = None
all_summaries = []

dataset_stats = {
    "videos_done": 0,
    "images_done": 0,
    "detections": 0,
    "misses": 0,
    "saved_rows": 0,
}


def init_display_areas():
    global live_handle, summary_handle, all_summaries, dataset_stats

    all_summaries = []
    dataset_stats = {
        "videos_done": 0,
        "images_done": 0,
        "detections": 0,
        "misses": 0,
        "saved_rows": 0,
    }

    live_handle = display(HTML("<pre>[LIVE] Waiting to start...</pre>"), display_id=True)
    summary_handle = display(HTML("<pre>[SUMMARY] No completed videos yet.</pre>"), display_id=True)


def update_live_status(
    stage_name,
    video_name,
    frame_idx,
    total_for_display,
    detected_count,
    missed_count,
    saved_count
):
    global live_handle, dataset_stats

    text = (
        "=" * 72 + "\n"
        f"[LIVE] Stage          : {stage_name}\n"
        f"[LIVE] Video          : {video_name}\n"
        f"[LIVE] Frame          : {frame_idx + 1}/{total_for_display}\n"
        f"[LIVE] Detected       : {detected_count}\n"
        f"[LIVE] Missed         : {missed_count}\n"
        f"[LIVE] Saved rows     : {saved_count}\n"
        + "-" * 72 + "\n"
        f"[DATASET] Videos done : {dataset_stats['videos_done']}\n"
        f"[DATASET] Images done : {dataset_stats['images_done']}\n"
        f"[DATASET] Detections  : {dataset_stats['detections']}\n"
        f"[DATASET] Misses      : {dataset_stats['misses']}\n"
        f"[DATASET] Saved rows  : {dataset_stats['saved_rows']}\n"
        + "=" * 72
    )

    live_handle.update(HTML(f"<pre>{text}</pre>"))


def append_video_summary(
    stage_name,
    video_name,
    output_csv_path,
    actual_frames_read,
    total_expected_images,
    processed_images,
    detected_count,
    missed_count,
    discarded_extra_frames
):
    global summary_handle, all_summaries

    diff = total_expected_images - actual_frames_read

    lines = []
    lines.append("=" * 72)
    lines.append(f"[SUMMARY] Stage               : {stage_name}")
    lines.append(f"[SUMMARY] Video               : {video_name}")
    lines.append(f"[SUMMARY] Frames read         : {actual_frames_read}")
    lines.append(f"[SUMMARY] Labels/images exp   : {total_expected_images}")
    lines.append(f"[SUMMARY] Images processed    : {processed_images}")
    lines.append(f"[SUMMARY] Discarded frames    : {discarded_extra_frames}")

    if diff == 0:
        lines.append(f"[SUMMARY] Status              : PERFECT MATCH ✅")
        lines.append(f"[SUMMARY] Extra labels/images : 0")
        lines.append(f"[SUMMARY] Extra frames        : 0")
    elif diff > 0:
        lines.append(f"[SUMMARY] Status              : EXTRA LABELS/IMAGES ⚠️")
        lines.append(f"[SUMMARY] Extra labels/images : {diff}")
        lines.append(f"[SUMMARY] Extra frames        : 0")
    else:
        lines.append(f"[SUMMARY] Status              : EXTRA FRAMES DISCARDED ⚠️")
        lines.append(f"[SUMMARY] Extra labels/images : 0")
        lines.append(f"[SUMMARY] Extra frames        : {-diff}")

    lines.append(f"[SUMMARY] Detections          : {detected_count}")
    lines.append(f"[SUMMARY] Missed detect       : {missed_count}")
    lines.append(f"[SUMMARY] Output CSV          : {output_csv_path}")
    lines.append("=" * 72)

    summary_text = "\n".join(lines)
    all_summaries.append(summary_text)
    combined = "\n\n".join(all_summaries)

    summary_handle.update(HTML(f"<pre>{combined}</pre>"))


def clear_live_area():
    global live_handle
    live_handle.update(HTML("<pre></pre>"))


def create_face_landmarker(model_path):
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1,
        min_face_detection_confidence=0.3,
        min_face_presence_confidence=0.3,
        min_tracking_confidence=0.3
    )
    return vision.FaceLandmarker.create_from_options(options)


LEFT_EYE_IDX = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDX = [263, 387, 385, 362, 373, 380]
MOUTH_IDX = [78, 81, 13, 311, 308, 402, 14, 178]


def safe_nan():
    return float("nan")


def is_valid_point(pt):
    if pt is None:
        return False
    if not isinstance(pt, (tuple, list)) or len(pt) != 2:
        return False

    x, y = pt
    if x is None or y is None:
        return False

    try:
        if math.isnan(x) or math.isnan(y):
            return False
    except TypeError:
        return False

    return True


def euclidean_dist(a, b):
    if not is_valid_point(a) or not is_valid_point(b):
        return safe_nan()
    return np.linalg.norm(np.array(a, dtype=np.float32) - np.array(b, dtype=np.float32))


def calculate_ear(landmarks, eye_indices):
    try:
        p1, p2, p3, p4, p5, p6 = [landmarks[i] for i in eye_indices]
    except (IndexError, TypeError):
        return safe_nan()

    distances = [
        euclidean_dist(p1, p4),
        euclidean_dist(p2, p6),
        euclidean_dist(p3, p5),
    ]

    if any(math.isnan(d) for d in distances):
        return safe_nan()

    den = distances[0]
    if den == 0:
        return safe_nan()

    ear = (distances[1] + distances[2]) / (2.0 * den)
    return float(ear)


def calculate_mar(landmarks, mouth_indices):
    try:
        p11, p12, p13, p14, p17, p18, p19, p20 = [landmarks[i] for i in mouth_indices]
    except (IndexError, TypeError):
        return safe_nan()

    den = euclidean_dist(p11, p17)
    if math.isnan(den) or den == 0:
        return safe_nan()

    d1 = euclidean_dist(p14, p18)
    d2 = euclidean_dist(p13, p19)
    d3 = euclidean_dist(p12, p20)

    if math.isnan(d1) or math.isnan(d2) or math.isnan(d3):
        return safe_nan()

    mar = (d1 + d2 + d3) / (3.0 * den)
    return float(mar)


def collect_image_files(folder_path):
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Image folder not found: {folder_path}")

    image_files = []
    for entry in sorted(os.scandir(folder_path), key=lambda e: e.name):
        if not entry.is_file():
            continue

        lower = entry.name.lower()
        if lower.endswith((".png", ".jpg", ".jpeg")):
            image_files.append(entry.path)

    return image_files


def extract_metrics_from_image(image_path, face_landmarker):
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")

    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Failed to read image file: {image_path}")

    h, w = image.shape[:2]

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_image
    )

    result = face_landmarker.detect(mp_image)

    if not result.face_landmarks or len(result.face_landmarks) == 0:
        return safe_nan(), safe_nan(), safe_nan()

    landmarks = result.face_landmarks[0]

    landmark_points = []
    for lm in landmarks:
        try:
            x = int(lm.x * w)
            y = int(lm.y * h)
            landmark_points.append((x, y))
        except Exception:
            landmark_points.append((safe_nan(), safe_nan()))

    left_ear = calculate_ear(landmark_points, LEFT_EYE_IDX)
    right_ear = calculate_ear(landmark_points, RIGHT_EYE_IDX)
    mar = calculate_mar(landmark_points, MOUTH_IDX)

    return left_ear, right_ear, mar


def parse_label_from_filename(image_path):
    stem = os.path.splitext(os.path.basename(image_path))[0]
    parts = stem.rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        return int(parts[1])
    return None


def all_metrics_nan(left_ear, right_ear, mar):
    return math.isnan(left_ear) and math.isnan(right_ear) and math.isnan(mar)


def process_image_folder(
    folder_path,
    output_csv_path,
    model_path,
    stage_name,
    display_name=None
):
    global dataset_stats

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    image_files = collect_image_files(folder_path)
    total_expected_images = len(image_files)

    shown_name = display_name if display_name is not None else os.path.basename(folder_path)

    print("=" * 72)
    print(f"[INFO] Processing folder : {folder_path}")
    print(f"[INFO] Output CSV        : {output_csv_path}")
    print(f"[INFO] Total images      : {total_expected_images}")
    print("=" * 72)

    if total_expected_images == 0:
        raise FileNotFoundError(f"No image files found in folder: {folder_path}")

    face_landmarker = create_face_landmarker(model_path)

    detected_count = 0
    missed_count = 0
    saved_count = 0
    processed_images = 0
    discarded_extra_frames = 0

    output_dir = os.path.dirname(output_csv_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    try:
        with open(output_csv_path, "w", encoding="utf-8") as f:
            f.write("image_path,left_ear,right_ear,mar,label\n")

            for frame_idx, image_path in enumerate(image_files):
                if not os.path.isfile(image_path):
                    raise FileNotFoundError(f"Image file not found: {image_path}")

                left_ear, right_ear, mar = extract_metrics_from_image(image_path, face_landmarker)
                label = parse_label_from_filename(image_path)

                if all_metrics_nan(left_ear, right_ear, mar):
                    missed_count += 1
                    dataset_stats["misses"] += 1
                else:
                    detected_count += 1
                    dataset_stats["detections"] += 1

                f.write(f"{image_path},{left_ear},{right_ear},{mar},{label}\n")
                saved_count += 1
                dataset_stats["saved_rows"] += 1

                processed_images += 1
                dataset_stats["images_done"] += 1

                update_live_status(
                    stage_name=stage_name,
                    video_name=shown_name,
                    frame_idx=frame_idx,
                    total_for_display=total_expected_images,
                    detected_count=detected_count,
                    missed_count=missed_count,
                    saved_count=saved_count
                )

    finally:
        face_landmarker.close()

    actual_frames_read = processed_images + discarded_extra_frames
    dataset_stats["videos_done"] += 1

    append_video_summary(
        stage_name=stage_name,
        video_name=shown_name,
        output_csv_path=output_csv_path,
        actual_frames_read=actual_frames_read,
        total_expected_images=total_expected_images,
        processed_images=processed_images,
        detected_count=detected_count,
        missed_count=missed_count,
        discarded_extra_frames=discarded_extra_frames
    )


def process_root_train(root_dir, output_root, model_path):
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"Train root directory not found: {root_dir}")

    for person_entry in sorted(os.scandir(root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name
        person_path = person_entry.path

        for condition_entry in sorted(os.scandir(person_path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            condition = condition_entry.name
            condition_path = condition_entry.path
            condition_clean = condition.split("_", 1)[-1] if "_" in condition else condition

            for video_entry in sorted(os.scandir(condition_path), key=lambda e: e.name):
                if not video_entry.is_dir():
                    continue

                video_name = video_entry.name
                video_folder = video_entry.path

                if not os.path.isdir(video_folder):
                    raise FileNotFoundError(f"Train video folder not found: {video_folder}")

                output_csv = os.path.join(
                    output_root,
                    "train_csv",
                    person_id,
                    condition_clean,
                    f"{video_name}.csv"
                )

                process_image_folder(
                    folder_path=video_folder,
                    output_csv_path=output_csv,
                    model_path=model_path,
                    stage_name="TRAIN",
                    display_name=video_name
                )


def process_root_validation(root_dir, output_root, model_path):
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"Validation root directory not found: {root_dir}")

    for person_entry in sorted(os.scandir(root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name
        person_path = person_entry.path

        for condition_entry in sorted(os.scandir(person_path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            condition = condition_entry.name
            condition_path = condition_entry.path

            for state_entry in sorted(os.scandir(condition_path), key=lambda e: e.name):
                if not state_entry.is_dir():
                    continue

                state = state_entry.name
                state_path = state_entry.path

                if not os.path.isdir(state_path):
                    raise FileNotFoundError(f"Validation state folder not found: {state_path}")

                display_name = f"{person_id}_{condition}_{state}"

                output_csv = os.path.join(
                    output_root,
                    "validation_csv",
                    person_id,
                    condition,
                    f"{state}.csv"
                )

                process_image_folder(
                    folder_path=state_path,
                    output_csv_path=output_csv,
                    model_path=model_path,
                    stage_name="VALIDATION",
                    display_name=display_name
                )


def process_root_test(root_dir, output_root, model_path):
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(f"Test root directory not found: {root_dir}")

    for person_entry in sorted(os.scandir(root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name
        person_path = person_entry.path

        for condition_entry in sorted(os.scandir(person_path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            condition = condition_entry.name
            condition_path = condition_entry.path

            for state_entry in sorted(os.scandir(condition_path), key=lambda e: e.name):
                if not state_entry.is_dir():
                    continue

                state = state_entry.name
                state_path = state_entry.path

                if not os.path.isdir(state_path):
                    raise FileNotFoundError(f"Test state folder not found: {state_path}")

                display_name = f"{person_id}_{condition}_{state}"

                output_csv = os.path.join(
                    output_root,
                    "test_csv",
                    person_id,
                    condition,
                    f"{state}.csv"
                )

                process_image_folder(
                    folder_path=state_path,
                    output_csv_path=output_csv,
                    model_path=model_path,
                    stage_name="TEST",
                    display_name=display_name
                )


MODEL_PATH = "face_landmarker.task"

TRAIN_ROOT_DIR = "/root/.cache/kagglehub/datasets/manith04/ddd-processed-2-training-frames-type-1/versions/1"
VALIDATION_ROOT_DIR = "/root/.cache/kagglehub/datasets/manith04/ddd-processed-validation-frames-type-1/versions/1"
TEST_ROOT_DIR = "/root/.cache/kagglehub/datasets/manith04/ddd-processed-test-frames-type-1/versions/1"

OUTPUT_ROOT = "/content/output"


if __name__ == "__main__":
    init_display_areas()

    process_root_train(TRAIN_ROOT_DIR, OUTPUT_ROOT, MODEL_PATH)
    process_root_validation(VALIDATION_ROOT_DIR, OUTPUT_ROOT, MODEL_PATH)
    process_root_test(TEST_ROOT_DIR, OUTPUT_ROOT, MODEL_PATH)

    clear_live_area()